# Airbnb NYC 2019 price modelling

The narrative version of `src/run_analysis.py`. Spatial features, three models at
increasing capacity, and an interactive Folium map saved alongside the static charts.

Before running: place `AB_NYC_2019.csv` at `data/AB_NYC_2019.csv`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

sns.set_style("whitegrid")
plt.rcParams.update({"figure.dpi": 110})

## 1. Load and filter

97 listings have prices at 0 or above 2,000 USD per night and look like data-entry
errors or misreported values. Drop them up front.

In [ ]:
df = pd.read_csv("data/AB_NYC_2019.csv")
print(f"rows: {len(df):,}")
df = df.loc[df["price"].between(1, 2000)].reset_index(drop=True)
print(f"after filter: {len(df):,}")
df[["neighbourhood_group", "room_type", "price", "minimum_nights", "availability_365"]].head()

## 2. Engineer landmark distances

Raw latitude/longitude work in a pinch, but landmark distances encode the
Manhattan-outward gradient far better. I used Haversine miles to five landmarks —
Times Square, Brooklyn Bridge, LGA, Central Park South, Prospect Park.

In [ ]:
LANDMARKS = {
    "times_square": (40.7580, -73.9855),
    "brooklyn_bridge": (40.7061, -73.9969),
    "lga_airport": (40.7769, -73.8740),
    "central_park_south": (40.7644, -73.9730),
    "prospect_park": (40.6602, -73.9690),
}

def haversine_miles(lat1, lon1, lat2, lon2):
    lat1r, lat2r = np.radians(lat1), np.radians(lat2)
    dlat = lat2r - lat1r
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1r) * np.cos(lat2r) * np.sin(dlon / 2) ** 2
    return 2 * 3958.8 * np.arcsin(np.sqrt(a))

for name, (lat, lon) in LANDMARKS.items():
    df[f"dist_{name}"] = haversine_miles(df["latitude"].values, df["longitude"].values, lat, lon)

df["reviews_per_month"] = df["reviews_per_month"].fillna(0)
df["log_price"] = np.log1p(df["price"])
df[["dist_times_square", "dist_brooklyn_bridge", "dist_lga_airport"]].describe().round(2)

## 3. Three models at increasing capacity

Baseline ridge (no distance features). Spatial ridge (adds the five distances and raw
coordinates). Gradient boosting (same feature set, non-linear).

In [ ]:
feature_cols = [
    "latitude", "longitude", "minimum_nights", "number_of_reviews",
    "reviews_per_month", "calculated_host_listings_count", "availability_365",
    "dist_times_square", "dist_brooklyn_bridge", "dist_lga_airport",
    "dist_central_park_south", "dist_prospect_park",
]
cat_cols = ["neighbourhood_group", "room_type"]
X = pd.get_dummies(df[feature_cols + cat_cols], columns=cat_cols, drop_first=True)
y = df["log_price"].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

baseline_cols = [c for c in X.columns
                 if c in feature_cols[:7]
                 or c.startswith("room_type_")
                 or c.startswith("neighbourhood_group_")]

rows = []
for name, fit_cols, model in [
    ("baseline_ridge", baseline_cols, Ridge(alpha=1.0)),
    ("spatial_ridge",   X.columns,     Ridge(alpha=1.0)),
    ("gradient_boost",  X.columns,     GradientBoostingRegressor(n_estimators=400, max_depth=3, learning_rate=0.05, random_state=42)),
]:
    model.fit(X_train[fit_cols], y_train)
    yhat = model.predict(X_test[fit_cols])
    rows.append({
        "model": name,
        "r2_log": r2_score(y_test, yhat),
        "mae_usd": mean_absolute_error(np.expm1(y_test), np.expm1(yhat)),
    })
pd.DataFrame(rows).round(3)

## 4. The interactive map

Folium produces an HTML file you can open in any browser. The heatmap shows price-
weighted listing density, and the markers sit on the ten most-expensive neighborhoods
that have at least 100 listings.

In [ ]:
heat = df.loc[df["price"].between(1, 500)].sample(n=15000, random_state=42)
m = folium.Map(location=[40.7359, -73.9911], zoom_start=11, tiles="CartoDB positron")
HeatMap(heat[["latitude", "longitude", "price"]].values.tolist(),
        min_opacity=0.3, radius=8, blur=10).add_to(m)
m  # renders in the notebook

## 5. What this isn't

A price-estimation project, not a yield prediction. An 800-USD listing that books ten
nights earns less than a 150-USD listing that books 200. Occupancy, cancellations,
cleaning fees, and seasonality all sit outside this snapshot.

And the dataset is 2019. Local Law 18 changed NYC short-term rental economics in
2023. A model trained on this dataset doesn't describe the 2026 market.

Full write-up: <https://ndjstn.github.io/posts/airbnb-nyc-price-room-type-geography/>.